In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torchvision.models as models

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# --- your paths ---
path = "C:\\Users\\alvin\\OneDrive\\Documents\\CS171\\project\\breast-cancer-classifier-cs171-s02"
bad_full = path + "/data/malignant"
good_full = path + "/data/benign"

source_dir = good_full  # 👈 using benign only

# --- model setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = torch.nn.Identity()  # remove classification layer
model = model.to(device)
model.eval()

transform = models.ResNet50_Weights.DEFAULT.transforms()

# --- extract embeddings ---
embeddings = []

print("Extracting deep features...")
for f in tqdm(os.listdir(source_dir)):
    if f.lower().endswith(('.jpg', '.png', '.jpeg')):
        path = os.path.join(source_dir, f)
        try:
            img = Image.open(path).convert("RGB")
            x = transform(img).unsqueeze(0).to(device)

            with torch.no_grad():
                emb = model(x).cpu().numpy().flatten()

            embeddings.append(emb)
        except:
            continue

X = np.array(embeddings)

# --- normalize ---
X_norm = StandardScaler().fit_transform(X)

# --- PCA for visualization ---
X_2d = PCA(n_components=2).fit_transform(X_norm)

# --- plot raw distribution ---
plt.figure(figsize=(6,4))
plt.scatter(X_2d[:,0], X_2d[:,1], alpha=0.4)
plt.title("PCA of deep features (benign only)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.show()

ModuleNotFoundError: No module named 'tqdm'